# S6E10 — 19 Our own TabPFN on an 8 GB GPU: how far does a reduced context go?

> **Pipeline position:** 08 TabPFN import (public members) → **19 own TabPFN probe** → 11 stack

| | |
|---|---|
| **Input** | `data/raw/train.csv`, `test.csv`; the public member `tabpfn_catfd` (`data/external/s6e10-tabpfn-member/`) for a same-rows comparison |
| **Output** | probe table; if a full fold 0 is run: `preds/tabpfn_own_folds/oof_0.npy` |
| **Runtime** | see the probe (RTX 3070 Ti, 8 GB) |
| **Result** | see *Results & decisions* |

**Why:** TabPFN carries the stack (weight 0.66), but we only have someone else's predictions (`08`). The author ran TabPFN-3.5 with the **whole fold as context (≈ 560k rows) on a machine with 121 GB of unified memory**. TabPFN does not train on our data; it *reads* labelled rows as context, and its AUC keeps growing with the context size. Our GPU has 8 GB, so the question is simply: **what context fits, and how much AUC is left at that size?**

**Setup — identical to the public member `tabpfn_catfd`** (fold 0 AUC 0.96117 with the full context): TabPFN-3.5 checkpoint, the 21 raw columns (categoricals as codes, missing arrival delay = −1), `Flight Distance` declared categorical, 1 estimator, int8 KV cache, autocast. The only difference is the context: a random subsample of the fold-0 training rows.

**Fair comparison:** the probe scores a fixed random subset of 20,000 fold-0 validation rows, and the public member is scored **on the same rows**.

> **Glossary:** *Context* = the labelled rows TabPFN reads before predicting (its "training set", without any weight updates). *KV cache* = the context's internal representation, computed once and reused for every prediction batch; its size grows with the context and is what limits the GPU. *int8* = 8-bit storage of that cache.

In [1]:
import torch          # first import: on the Mac the kernel crashes if torch comes after numpy/sklearn (OpenMP conflict)
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import time, gc, warnings
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion
import importlib.metadata
warnings.filterwarnings('ignore')

DEVICE = 'cuda'
RUN_PROBE = False     # True: rerun the context probe of section 1 (~10 min); its results are in the markdown below
CTX_SIZES = [25_000, 50_000, 100_000, 200_000]     # tried in order until the GPU runs out of memory
N_PROBE = 20_000                                   # validation rows scored in the probe
BATCH = 4096                                       # rows per prediction call (prediction is the memory peak)
RAW = next(p for p in [Path('data/raw'), Path('../data/raw')] if (p / 'train.csv').exists())
EXT = next(p for p in [Path('data/external/s6e10-tabpfn-member'), Path('../data/external/s6e10-tabpfn-member')] if p.exists())
PREDS_DIR = next(p for p in [Path('preds'), Path('../preds')] if p.exists())
print('tabpfn', importlib.metadata.version('tabpfn'), '| GPU', torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB')

train = pd.read_csv(RAW / 'train.csv'); test = pd.read_csv(RAW / 'test.csv')
CATS = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
FEATS = [c for c in test.columns if c != 'id']
y = train['satisfaction'].astype(str).str.strip().str.lower().isin(['true', '1', 'satisfied']).astype(int).values
full = pd.concat([train[FEATS], test[FEATS]], ignore_index=True)
for c in CATS: full[c] = full[c].astype('category').cat.codes
full['Arrival Delay in Minutes'] = full['Arrival Delay in Minutes'].fillna(-1)
X = full.values.astype(np.float32); Xtr, Xte = X[:len(train)], X[len(train):]
cat_idx = [full.columns.get_loc(c) for c in CATS] + [full.columns.get_loc('Flight Distance')]   # "catfd": distance as a route id
FOLDS = list(StratifiedKFold(5, shuffle=True, random_state=42).split(Xtr, y))
tr0, va0 = FOLDS[0]
probe_rows = np.sort(np.random.RandomState(0).choice(va0, N_PROBE, replace=False))
pub = pd.read_csv(EXT / 'oof_tabpfn_catfd.csv').set_index('id')['pred'].reindex(train['id']).values
print(f'public tabpfn_catfd (full context): fold 0 {roc_auc_score(y[va0], pub[va0]):.5f} | on the {N_PROBE:,} probe rows {roc_auc_score(y[probe_rows], pub[probe_rows]):.5f}')

tabpfn 9.1.0 | GPU NVIDIA GeForce RTX 3070 Ti 8.0 GB


public tabpfn_catfd (full context): fold 0 0.96117 | on the 20,000 probe rows 0.96120


## 1. Probe: growing context on fold 0
Each size: subsample the context, fit (builds the KV cache), predict the probe rows in batches. Peak GPU memory and times are recorded. The loop stops at the first out-of-memory error.

In [2]:
def make_model():
    return TabPFNClassifier.create_default_for_version(
        ModelVersion.V3_5, device=DEVICE, n_estimators=1, random_state=0, ignore_pretraining_limits=True,
        fit_mode='fit_with_cache', kv_cache_precision='int8', inference_precision='autocast',
        memory_saving_mode='auto', categorical_features_indices=cat_idx)

def predict(m, Xq):
    out = np.empty(len(Xq), dtype=np.float32)
    for i in range(0, len(Xq), BATCH): out[i:i + BATCH] = m.predict_proba(Xq[i:i + BATCH])[:, 1]
    return out

rows = []
for ctx in (CTX_SIZES if RUN_PROBE else []):
    ctx_idx = np.random.RandomState(0).choice(tr0, ctx, replace=False)     # context = labelled rows of the fold's TRAINING part only
    try:
        torch.cuda.reset_peak_memory_stats(); t0 = time.time()
        m = make_model(); m.fit(Xtr[ctx_idx], y[ctx_idx]); t_fit = time.time() - t0
        t0 = time.time(); p = predict(m, Xtr[probe_rows]); t_pred = time.time() - t0
        rows.append({'context': ctx, 'probe AUC': roc_auc_score(y[probe_rows], p), 'fit s': round(t_fit), 'predict s / 20k': round(t_pred),
                     'peak GB': round(torch.cuda.max_memory_allocated() / 2**30, 2),
                     'est. min for 140k val + 300k test': round(t_pred / N_PROBE * 440_000 / 60, 1)})
        print(rows[-1], flush=True)
    except torch.OutOfMemoryError as e:
        print(f'context {ctx:,}: OUT OF MEMORY -> stop', flush=True); break
    finally:
        m = None; gc.collect(); torch.cuda.empty_cache()
probe = pd.DataFrame(rows)
if len(probe): probe['vs public (same rows)'] = probe['probe AUC'] - roc_auc_score(y[probe_rows], pub[probe_rows])
print(probe.round(5).to_string(index=False) if len(probe) else 'probe skipped (RUN_PROBE = False)')

probe skipped (RUN_PROBE = False)


**Probe results (run once, 2026-10-10):**

| Context | Probe AUC | vs public (0.96120, same rows) | Fit | Predict 20k | Peak GPU memory |
|---|---|---|---|---|---|
| 25k | 0.95791 | −0.00329 | 4 s | 2 s | 3.6 GB |
| 50k | 0.95910 | −0.00210 | 11 s | 4 s | 6.4 GB |
| 100k | 0.95927 | −0.00193 | 38 s | 7 s | 8.8 GB |
| 200k | 0.96006 | −0.00113 | 170 s | 13 s | 9.1 GB (spills from the 8 GB VRAM into shared memory, still fast) |

AUC keeps growing with the context; 200k fits.

## 2. Our features at a 200k context (fold 0, full validation)
The reason to run TabPFN ourselves: we can give it **our** information from other rows and from outside, which is what moved the score in this project. Two variants on the **same 200k context rows**, both scored on all fold-0 validation rows:

| Variant | Columns |
|---|---|
| `catfd` | as the public member: 21 raw columns, `Flight Distance` categorical |
| `ours` | `catfd` + in-fold TE of the route and of route × class + the two captains `orig_logit` (03) and `og_cat_logit` (09) |

**Leakage guard:** the target encoder is fitted on the fold-0 training part with inner 5-fold cross-fitting (the context rows are never encoded with their own label) and only *applied* to the validation rows. The captains are trained on the original survey only.

In [3]:
from sklearn.preprocessing import TargetEncoder
from sklearn.linear_model import LogisticRegression
PROC = next(p for p in [Path('data/processed'), Path('../data/processed')] if (p / 'X_rich_train.parquet').exists())
CTX = 200_000
ctx_idx = np.random.RandomState(0).choice(tr0, CTX, replace=False)       # the same context rows as the 200k probe
fd = full['Flight Distance'].values[:len(train)].astype(int); cls = full['Class'].values[:len(train)].astype(int)
keys = pd.DataFrame({'fd': fd, 'fd_cls': fd * 10 + cls})
TE = np.full((len(train), 2), np.nan, dtype=np.float32)
enc = TargetEncoder(target_type='binary', cv=5, shuffle=True, random_state=42)
TE[tr0] = enc.fit_transform(keys.iloc[tr0], y[tr0])        # cross-fitted on the fold's training part
TE[va0] = enc.transform(keys.iloc[va0])                    # validation: only transformed
capt = np.column_stack([pd.read_parquet(PROC / 'X_final_train.parquet', columns=['orig_logit'])['orig_logit'].values,
                        pd.read_parquet(PROC / 'X_rich_train.parquet', columns=['og_cat_logit'])['og_cat_logit'].values]).astype(np.float32)
VARIANTS = {'catfd': Xtr, 'ours': np.hstack([Xtr, TE, capt])}

P0 = {}
for name, XX in VARIANTS.items():
    d = PREDS_DIR / f'tabpfn_own_{name}_ctx200k_folds'; f = d / 'oof_0.npy'
    if f.exists():
        P0[name] = np.load(f); print(f'{name}: loaded'); continue
    t0 = time.time(); m = make_model(); m.fit(XX[ctx_idx], y[ctx_idx])
    P0[name] = predict(m, XX[va0]); m = None; gc.collect(); torch.cuda.empty_cache()
    d.mkdir(exist_ok=True); np.save(f, P0[name])
    print(f'{name}: fold 0 AUC {roc_auc_score(y[va0], P0[name]):.5f} ({(time.time() - t0) / 60:.1f} min)', flush=True)
print(f"\npublic tabpfn_catfd (560k context) fold 0 {roc_auc_score(y[va0], pub[va0]):.5f}")
for name in P0: print(f'{name:6s} (200k context)        fold 0 {roc_auc_score(y[va0], P0[name]):.5f}')

catfd: fold 0 AUC 0.96030 (4.8 min)


ours: fold 0 AUC 0.96061 (4.6 min)



public tabpfn_catfd (560k context) fold 0 0.96117
catfd  (200k context)        fold 0 0.96030
ours   (200k context)        fold 0 0.96061


## 3. Mini-stack on the rows of fold 0
As in `15`/`17`: LR (C = 10) on the logits of the 10 current members, inner 5-fold CV over the fold-0 rows, + each variant.

In [4]:
lg = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
as_logit = lambda o: lg(o) if (o.min() >= 0 and o.max() <= 1) else o
NAMES = {'cat_v1': 'cat_v1', 'cat_v2': 'cat_v2', 'xgb': 'xgb', 'lgbm_tuned': 'lgbm_tuned_final', 'mlp': 'mlp', 'realmlp': 'realmlp_avg',
         'realmlp_rich': 'realmlp_rich_avg', 'tabpfn': 'tabpfn_avg', 'tabpfn_raw': 'tabpfn_raw_avg', 'tabicl': 'tabicl'}
B = np.column_stack([as_logit(np.load(PREDS_DIR / f'oof_{s}.npy').astype(float))[va0] for s in NAMES.values()]); yv = y[va0]
inner = list(StratifiedKFold(5, shuffle=True, random_state=42).split(B, yv))
score = lambda M: np.array([roc_auc_score(yv[b], M[b] @ LogisticRegression(C=10.0, max_iter=500).fit(M[a], yv[a]).coef_[0]) for a, b in inner])
base = score(B); rows = [{'stack': '10 members', 'mean AUC': base.mean(), 'diff': 0.0, 'positive': '-', 'weight of new': None}]
for name in P0:
    M = np.hstack([B, lg(P0[name])[:, None]]); s_ = score(M); d_ = s_ - base
    w = LogisticRegression(C=10.0, max_iter=500).fit(M, yv).coef_[0][-1]
    rows.append({'stack': f'+ own TabPFN {name}', 'mean AUC': s_.mean(), 'diff': d_.mean(), 'positive': f'{(d_ > 0).sum()}/5', 'weight of new': round(w, 3)})
print(pd.DataFrame(rows).round(5).to_string(index=False))
print('Spearman with public tabpfn_avg:', {n: round(pd.Series(lg(P0[n])).corr(pd.Series(B[:, list(NAMES).index('tabpfn')]), method='spearman'), 4) for n in P0})

             stack  mean AUC     diff positive  weight of new
        10 members   0.96193  0.00000        -            NaN
+ own TabPFN catfd   0.96191 -0.00001      2/5         -0.008
 + own TabPFN ours   0.96192 -0.00000      1/5          0.029
Spearman with public tabpfn_avg: {'catfd': np.float64(0.9709), 'ours': np.float64(0.9726)}


## Results & decisions

**Fold 0, all 140k validation rows, TabPFN-3.5 on an RTX 3070 Ti:**

| Run | Context | Fold 0 AUC | Time |
|---|---|---|---|
| public `tabpfn_catfd` (`08`) | 560k (121 GB machine) | 0.96117 | – |
| own `catfd` | 200k | 0.96030 | 4.8 min |
| **own `ours`** (+ route TE, route × class TE, 2 captains) | 200k | **0.96061** | 4.6 min |

**Mini-stack on the fold-0 rows** (10 members = 0.96193): + own `catfd` −0.00001 (2/5), + own `ours` −0.00000 (1/5); stack weights −0.008 and 0.029.

**Decision: closed.** Our features do help TabPFN itself (+0.0003 at the same context), and a 200k context fits an 8 GB GPU better than expected (spilling into shared memory without a large slowdown). But the stack gives the new member almost zero weight: what our TabPFN knows is already covered by the public TabPFN average (Spearman 0.97) and by RealMLP-rich, which gets the same TE and captain features. A full-context run (≈ 25 min fit per fold, memory beyond VRAM) would raise TabPFN's own AUC by ≈ 0.001 at most — it would have to add far more than that to the stack, which these numbers do not suggest. Not pursued.

**For the retrospective:** an 8 GB consumer GPU runs TabPFN-3.5 with a 200k-row context (int8 KV cache); AUC still grows with context at that size (25k → 200k: 0.9579 → 0.9601). On Windows, tabpfn 9.1.0's browser licence flow crashes (`select` on stdin); saving the API key with `tabpfn.browser_auth.save_token` works around it.


## Glossary
* **Context** — the labelled rows TabPFN reads before predicting.
* **KV cache** — the stored representation of the context; it is what has to fit into GPU memory.
* **Probe rows** — a fixed random subset of fold-0 validation rows, scored for both our run and the public member.